# se-path-multi-v2

Historical `se-path-multi-v1` is not modified. This run optimises clipped synthetic mastery change, not the v1 quiz correctness bonus.
Training revision `se-path-multi-v2-coverage-r2`: every seed alternates Y3S1 and Y3S2. An even seed is not Y3S1-only, and an odd seed is not Y3S2-only.
Primary reported outcome is `mean_mastery_change`. Return is not accuracy.
Keep `RUN_FULL_TRAINING = False` until the intentional Colab run.
The notebook installs dependencies, uploads `se_path_multi_v2_colab.zip`, and finds the source root. It does not expect `/content/c4`.
Smoke writes a new timestamped folder. The full run also creates a new timestamped folder and does not overwrite an older `run_full` directory.


In [ ]:
%pip install -q numpy gymnasium==1.4.0 stable-baselines3==2.9.0 sb3-contrib==2.9.0


In [ ]:
RUN_FULL_TRAINING = False
print('RUN_FULL_TRAINING', RUN_FULL_TRAINING)


In [ ]:
import shutil
import sys
import zipfile
from pathlib import Path

MARKER = Path('training_se_v2') / 'multi_action_v2' / 'smoke_runner.py'

def find_zip() -> Path:
    found = sorted(Path('/content').glob('se_path_multi_v2_colab.zip'))
    if found:
        return found[0]
    from google.colab import files
    uploaded = files.upload()
    zips = [Path('/content') / name for name in uploaded if name.endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Upload se_path_multi_v2_colab.zip')
    return zips[0]

def find_source_root(start: Path) -> Path:
    if (start / MARKER).is_file():
        return start
    for path in start.rglob('smoke_runner.py'):
        if path.parent.name == 'multi_action_v2' and path.parent.parent.name == 'training_se_v2':
            return path.parents[2]
    raise FileNotFoundError(f'{MARKER} was not found in the extracted ZIP')

zip_path = find_zip()
extract_to = Path('/content/se_path_multi_v2_src')
if extract_to.exists():
    shutil.rmtree(extract_to)
extract_to.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_to)
ROOT = find_source_root(extract_to)
protocol_file = ROOT / 'training_se_v1' / 'protocol.py'
objective_file = ROOT / 'training_se_v2' / 'multi_action_v2' / 'objective.py'
if not protocol_file.is_file() or not objective_file.is_file():
    raise FileNotFoundError('extracted ZIP is missing protocol.py or objective.py')
sys.path[:] = [str(ROOT)] + [entry for entry in sys.path if entry not in ('', str(ROOT))]
print('ZIP', zip_path)
print('ROOT', ROOT)


In [ ]:
import training_se_v1.protocol as se_protocol
from training_se_v2.multi_action_v2.smoke_runner import main
print('protocol module', se_protocol.__file__)
if not str(se_protocol.__file__).startswith(str(ROOT)):
    raise RuntimeError('protocol resolved outside the extracted ZIP')
main()


In [ ]:
if 'ROOT' not in dir():
    raise RuntimeError('Run the ZIP extraction cell first.')
if not RUN_FULL_TRAINING:
    print('FULL TRAINING SKIPPED. Smoke does not approve a policy.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    from training_se_v2.multi_action_v2.full_run import run_full
    parent = Path('/content/drive/MyDrive/AdaptiveLearnSE/c4_runs/se_path_multi_v2')
    manifest = run_full(parent, timesteps=80_000, episodes=20)
    print('training_revision', manifest['training_revision'])
    print('semester_coverage', manifest['semester_coverage'])
    print('criterion_met', manifest['criterion_met'])
    print('activation_allowed', manifest['activation_allowed'])
    print('serves_students', manifest['serves_students'])
    print('run_dir', manifest['run_dir'])
    print('export', Path(manifest['run_dir']) / 'export')


## Selection and export

Seeds 0, 1 and 2. Each seed trains on both semesters. Select the highest validation `mean_mastery_change` across Y3S1 and Y3S2 on the shared bank (`81000` / `82000`). Do not use the held-out bank (`91000` / `92000`) while choosing.
A full export without both semesters for every seed is rejected. `criterion_met` is still the mastery and invalid-action floor. It is not lowered.
If `criterion_met` is false, do not import the export.
If it is true, copy that run's `export/` to `ai-services/c4/runtime-artifacts/se_path_multi_v2/active/` only. The student plan does not call this endpoint.
